# EAAR-NLU V1-lite (3 of 6 layers): does the middle section help?

**Layers kept:** 1 Semantic Evidence Construction → 2 Typed Slot Encoding → 5 Visual-Semantic Cross-Attention, injected through the gated residual (the gate is what keeps the E0-R baseline intact).
**Layers skipped for now:** 3 Role Binding, 4 Reliability Weighting (switches already in the code, so the full model is one flag away).

**Experiment design (all on the same validation images, same decoding settings):**

| Row | What it is | Why |
|---|---|---|
| E0-R baseline | frozen E0-R, no detections | the number to beat |
| 3-part bridge | trained, real detections | the claimed improvement |
| 3-part, detections shuffled at test time | same weights, wrong image's detections | if the score drops, the model really uses grounded evidence |
| 3-part, trained on shuffled detections | control: same parameters, meaningless evidence | separates "grounded semantics" from "just 2.8M extra parameters" |

Outputs saved to `/kaggle/working/e1_outputs/`: `results.csv/json`, `progress_curve.png`, `results_bar.png`, `qualitative.png`.

Aligned with the E0-R evaluation (same 1,619 val images, decoding, raw-string metrics; baseline should give CIDEr ≈ 0.4696).

Needs: GPU, Internet, and your **E0-R `ckpt_epoch05.pt` attached as an input dataset**.

In [47]:
import os, sys, subprocess
REPO = "/kaggle/working/image_captioning"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "https://github.com/aditi2024-2028/image_captioning.git", REPO], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
os.makedirs(f"{REPO}/tests", exist_ok=True)
print(os.getcwd())

/kaggle/working/image_captioning


In [48]:
!pip -q install pycocoevalcap pytest

In [49]:
import glob, re
SMOKE_TEST = False          # True = 800 train / 200 val images, 40 steps. Set False for the real run.

DATASET   = "/kaggle/input/datasets/adityajn105/flickr8k"
IMAGE_DIR, ANN_FILE = DATASET + "/Images", DATASET + "/captions.txt"
OUT_DIR   = "/kaggle/working/e1_outputs"; os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
EPOCHS, BATCH, ACCUM = 3, 4, 4
LR, WEIGHT_DECAY, WARMUP, MAX_GRAD = 3e-4, 0.01, 100, 1.0
AMP = True
N_CURVE = 500                                   # val images used for the per-epoch progress curve
GEN_KW = dict(num_beams=4, max_new_tokens=30, min_length=5, length_penalty=1.0, no_repeat_ngram_size=3, early_stopping=True)  # identical to evaluate.py
GATE_LR = 3e-3                                   # tanh(gate) starts at 0; give it a larger lr so the bridge can open

LITE = dict(use_role_binding=False, use_reliability=False)       # the 3-part model
VARIANTS = {                                                     # name -> (bridge kwargs, train-time detection mode)
    "3part":                (LITE, "real"),
    "3part_shuffled_train": (LITE, "shuffled"),                  # control; delete this line to save time
    # "full6":              ({}, "real"),                        # later: the full 6-layer model
}

if SMOKE_TEST:
    N_TRAIN, N_VAL, MAX_OPT_STEPS, EPOCHS, N_CURVE = 800, 200, 40, 1, 100
else:
    N_TRAIN, N_VAL, MAX_OPT_STEPS = None, None, None

# ---- E0-R checkpoint: always epoch 05 (the one the peers evaluated: CIDEr 0.4696) ----
E0R_CKPT = "/kaggle/working/ckpt_epoch05.pt"                                  # or set a path by hand
if E0R_CKPT is None:
    own = f"{REPO}/outputs/e0r/ckpt_epoch05.pt"
    found = [own] if os.path.exists(own) else []
    found += glob.glob("/kaggle/input/**/ckpt_epoch05.pt", recursive=True)
    print("epoch-05 checkpoints found:", *found, sep="\n  ")
    E0R_CKPT = found[0] if found else None
if E0R_CKPT is None:
    raise FileNotFoundError("ckpt_epoch05.pt not found. Attach the dataset that contains it (+ Add Input) or set E0R_CKPT by hand.")
print("USING:", E0R_CKPT)
PEER_REF = {"CIDEr": 0.4696, "BLEU-4": 0.182283, "ROUGE-L": 0.497656}   # from the peers' evaluate.py run on ckpt_epoch05

USING: /kaggle/working/ckpt_epoch05.pt


## Module files and unit tests (expect **9 passed**)

In [50]:
%%writefile /kaggle/working/image_captioning/src/models/eaar_nlu.py
"""
EAAR-NLU V1 bridge
==================
Six trainable layers that sit between the frozen perception front-end
(CLIP ViT-B/16 tokens + RF-DETR Small detections) and the frozen BART decoder.

    tokens  [B, 197, 768]  ─┐
                            ├─►  1 Semantic Evidence Construction  [B, 28, 768]  (+ slot mask)
    detections (<= 8 boxes) ┘    2 Typed Slot Encoding             [B, 28, 768]
                                 3 Role Binding (masked self-attn)  [B, 28, 768]
                                 4 Reliability Weighting            [B, 28, 1] -> weighted slots
                                 5 Visual-Semantic Cross-Attention  [B, 197, 768]
                                 6 Gated Residual Fusion            [B, 197, 768]

Slot layout (28 = 3 * 8 + 4), fixed positions:
    [0  : 8 )  entity      one per detection
    [8  : 16)  attribute   one per detection
    [16 : 20)  action      top-4 detections by confidence (+ global CLS context)
    [20 : 28)  relation    detection i <-> its nearest other detection

Empty slots are carried as zeros with ``mask == False`` and are ignored by every
attention layer. Samples with no detections at all pass through unchanged.

Detection boxes are expected in the frame of the 224x224 image the CLIP encoder
actually sees (normalised xyxy in [0, 1]); ``detections_to_tensors`` converts
RF-DETR output from the original image into that frame.
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Optional, Sequence

import torch
import torch.nn as nn
from torch import Tensor
from torchvision.ops import roi_align

SLOT_ENTITY, SLOT_ATTRIBUTE, SLOT_ACTION, SLOT_RELATION = 0, 1, 2, 3


# ---------------------------------------------------------------------------
# Detection container + RF-DETR adapter
# ---------------------------------------------------------------------------

@dataclass
class Detections:
    """Padded detections. Per-sample tensors are [K, ...]; batched are [B, K, ...].

    boxes  : float  [.., K, 4]  normalised xyxy in the 224x224 CLIP frame
    labels : long   [.., K]     COCO category id (RF-DETR class_id)
    scores : float  [.., K]     detection confidence
    valid  : bool   [.., K]     False for padding
    """

    boxes: Tensor
    labels: Tensor
    scores: Tensor
    valid: Tensor

    def to(self, device, non_blocking: bool = True) -> "Detections":
        return Detections(
            self.boxes.to(device, non_blocking=non_blocking),
            self.labels.to(device, non_blocking=non_blocking),
            self.scores.to(device, non_blocking=non_blocking),
            self.valid.to(device, non_blocking=non_blocking),
        )

    @staticmethod
    def collate(items: Sequence["Detections"]) -> "Detections":
        return Detections(
            torch.stack([d.boxes for d in items]),
            torch.stack([d.labels for d in items]),
            torch.stack([d.scores for d in items]),
            torch.stack([d.valid for d in items]),
        )

    @staticmethod
    def empty(k: int = 8) -> "Detections":
        return Detections(
            torch.zeros(k, 4),
            torch.zeros(k, dtype=torch.long),
            torch.zeros(k),
            torch.zeros(k, dtype=torch.bool),
        )


def detections_to_tensors(
    xyxy,
    confidence,
    class_id,
    image_size: tuple[int, int],
    max_det: int = 8,
    out_size: int = 224,
    min_conf: float = 0.0,
    min_area_frac: float = 1e-3,
) -> Detections:
    """Convert RF-DETR output (original-image pixels) to a padded ``Detections``.

    CLIP's preprocessing resizes the short side to ``out_size`` and centre-crops, so
    boxes must go through the same transform; boxes that fall outside the crop are
    clipped and dropped if almost nothing is left.

    xyxy        : array-like [N, 4] pixel coordinates in the ORIGINAL image
    confidence  : array-like [N]
    class_id    : array-like [N]
    image_size  : (width, height) of the original image
    """
    w, h = image_size
    conf = torch.as_tensor(confidence, dtype=torch.float32).reshape(-1)
    cls = torch.as_tensor(class_id, dtype=torch.long).reshape(-1)
    boxes = torch.as_tensor(xyxy, dtype=torch.float32).reshape(conf.numel(), 4)

    scale = out_size / float(min(w, h))
    off_x = (w * scale - out_size) / 2.0
    off_y = (h * scale - out_size) / 2.0
    boxes = boxes * scale
    boxes = torch.stack(
        [boxes[:, 0] - off_x, boxes[:, 1] - off_y, boxes[:, 2] - off_x, boxes[:, 3] - off_y],
        dim=1,
    )
    boxes = boxes.clamp(0.0, float(out_size)) / out_size

    area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    keep = (area >= min_area_frac) & (conf >= min_conf)
    boxes, conf, cls = boxes[keep], conf[keep], cls[keep]

    order = conf.argsort(descending=True)[:max_det]
    boxes, conf, cls = boxes[order], conf[order], cls[order]
    n = boxes.shape[0]

    out = Detections.empty(max_det)
    out.boxes[:n], out.scores[:n], out.labels[:n] = boxes, conf, cls
    out.valid[:n] = True
    return out


def _sort_and_fit(det: Detections, k: int):
    """Sort by confidence (invalid last) and pad / truncate to exactly k entries."""
    scores = det.scores.masked_fill(~det.valid, -1.0)
    order = scores.argsort(dim=1, descending=True)
    boxes = det.boxes.gather(1, order.unsqueeze(-1).expand(-1, -1, 4))
    labels = det.labels.gather(1, order)
    scores = det.scores.gather(1, order)
    valid = det.valid.gather(1, order)

    def fit(t: Tensor, fill) -> Tensor:
        if t.shape[1] >= k:
            return t[:, :k]
        pad = t.new_full((t.shape[0], k - t.shape[1], *t.shape[2:]), fill)
        return torch.cat([t, pad], dim=1)

    return fit(boxes, 0.0), fit(labels, 0), fit(scores, 0.0), fit(valid, False)


# ---------------------------------------------------------------------------
# Shared building blocks
# ---------------------------------------------------------------------------

def _mlp(dim: int, hidden: int) -> nn.Sequential:
    return nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))


class MaskedMHA(nn.Module):
    """Multi-head scaled dot-product attention with a key padding mask.

    ``key_mask`` is bool [B, Lk] (True = real). Samples whose keys are all masked
    return zeros instead of NaN.
    """

    def __init__(self, dim: int, attn_dim: int, num_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        assert attn_dim % num_heads == 0, "attn_dim must be divisible by num_heads"
        self.h = num_heads
        self.hd = attn_dim // num_heads
        self.q = nn.Linear(dim, attn_dim)
        self.k = nn.Linear(dim, attn_dim)
        self.v = nn.Linear(dim, attn_dim)
        self.o = nn.Linear(attn_dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x_q: Tensor, x_kv: Tensor, key_mask: Tensor, return_weights: bool = False):
        B, Lq, _ = x_q.shape
        Lk = x_kv.shape[1]
        q = self.q(x_q).view(B, Lq, self.h, self.hd).transpose(1, 2)
        k = self.k(x_kv).view(B, Lk, self.h, self.hd).transpose(1, 2)
        v = self.v(x_kv).view(B, Lk, self.h, self.hd).transpose(1, 2)

        any_valid = key_mask.any(dim=-1)                       # [B]
        safe = key_mask.clone()
        safe[:, 0] |= ~any_valid                               # avoid all -inf rows

        logits = (q @ k.transpose(-1, -2)) / math.sqrt(self.hd)  # [B, H, Lq, Lk]
        logits = logits.masked_fill(~safe[:, None, None, :], torch.finfo(logits.dtype).min)
        attn = self.drop(logits.softmax(dim=-1))

        ctx = (attn @ v).transpose(1, 2).reshape(B, Lq, self.h * self.hd)
        out = self.o(ctx) * any_valid[:, None, None].to(ctx.dtype)
        return (out, attn) if return_weights else out


# ---------------------------------------------------------------------------
# Layer 1 - Semantic Evidence Construction
# ---------------------------------------------------------------------------

class SemanticEvidenceConstruction(nn.Module):
    """Detections + CLIP patch grid -> entity / attribute / action / relation candidates."""

    def __init__(
        self,
        dim: int,
        num_labels: int,
        max_entities: int,
        num_action_slots: int,
        grid_size: int,
        bottleneck_dim: int,
    ) -> None:
        super().__init__()
        assert num_action_slots <= max_entities
        self.dim, self.k, self.a, self.g = dim, max_entities, num_action_slots, grid_size
        self.num_labels = num_labels

        self.label_emb = nn.Embedding(num_labels, dim)
        self.geo_proj = nn.Linear(8, dim)     # x1 y1 x2 y2 conf w h area
        self.pair_proj = nn.Linear(6, dim)    # dx dy log-w-ratio log-h-ratio iou dist
        self.attr_mlp = _mlp(dim, bottleneck_dim)
        self.act_mlp = _mlp(dim, bottleneck_dim)
        self.rel_mlp = _mlp(dim, bottleneck_dim)
        nn.init.normal_(self.label_emb.weight, std=0.02)

    @property
    def num_slots(self) -> int:
        return 3 * self.k + self.a

    def forward(self, tokens: Tensor, det: Detections):
        B, N, d = tokens.shape
        G, K, A = self.g, self.k, self.a
        assert N == 1 + G * G, f"expected {1 + G * G} tokens, got {N}"

        boxes, labels, scores, valid = _sort_and_fit(det, K)
        boxes = boxes.float()
        scores = scores.float()

        # ---- region features: RoIAlign over the 14x14 patch grid -----------
        grid = tokens[:, 1:, :].transpose(1, 2).reshape(B, d, G, G)
        ms = 1.0 / G
        dummy = boxes.new_tensor([0.0, 0.0, 1.0, 1.0])
        rb = torch.where(valid.unsqueeze(-1), boxes, dummy)
        x1 = rb[..., 0].clamp(0.0, 1.0 - ms)
        y1 = rb[..., 1].clamp(0.0, 1.0 - ms)
        x2 = torch.maximum(rb[..., 2], x1 + ms).clamp(max=1.0)
        y2 = torch.maximum(rb[..., 3], y1 + ms).clamp(max=1.0)
        rois = torch.stack([x1, y1, x2, y2], dim=-1).reshape(B * K, 4) * G
        bidx = torch.arange(B, device=tokens.device).repeat_interleave(K).unsqueeze(1)
        rois = torch.cat([bidx.to(rois.dtype), rois], dim=1).to(grid.dtype)
        region = roi_align(grid, rois, output_size=2, spatial_scale=1.0,
                           sampling_ratio=2, aligned=True)      # [B*K, d, 2, 2]
        region = region.mean(dim=(2, 3)).reshape(B, K, d)

        # ---- geometry --------------------------------------------------------
        gb = boxes * valid.unsqueeze(-1)
        gx1, gy1, gx2, gy2 = gb.unbind(-1)
        w = (gx2 - gx1).clamp(min=0.0)
        h = (gy2 - gy1).clamp(min=0.0)
        cx, cy = (gx1 + gx2) / 2, (gy1 + gy2) / 2
        geom = torch.stack([gx1, gy1, gx2, gy2, scores * valid, w, h, w * h], dim=-1)

        lab = self.label_emb(labels.clamp(0, self.num_labels - 1))

        # ---- entity / attribute / action ------------------------------------
        ent = region + lab + self.geo_proj(geom.to(tokens.dtype))
        attr = self.attr_mlp(region + lab)
        act = self.act_mlp(region[:, :A] + lab[:, :A] + tokens[:, :1])

        # ---- relation: each entity paired with its nearest other entity -----
        centers = torch.stack([cx, cy], dim=-1)
        dist = torch.cdist(centers, centers)                              # [B, K, K]
        eye = torch.eye(K, dtype=torch.bool, device=tokens.device).unsqueeze(0)
        pair_ok = valid.unsqueeze(2) & valid.unsqueeze(1) & ~eye
        nn_dist, nn_idx = dist.masked_fill(~pair_ok, 1e4).min(dim=-1)     # [B, K]
        rel_valid = valid & (nn_dist < 1e3)

        ent_j = ent.gather(1, nn_idx.unsqueeze(-1).expand(-1, -1, d))
        nb = gb.gather(1, nn_idx.unsqueeze(-1).expand(-1, -1, 4))
        nx1, ny1, nx2, ny2 = nb.unbind(-1)
        nw, nh = (nx2 - nx1).clamp(min=0.0), (ny2 - ny1).clamp(min=0.0)
        ncx, ncy = (nx1 + nx2) / 2, (ny1 + ny2) / 2
        eps = 1e-3
        dx = ((ncx - cx) / w.clamp(min=eps)).clamp(-8.0, 8.0)
        dy = ((ncy - cy) / h.clamp(min=eps)).clamp(-8.0, 8.0)
        lw = torch.log(nw.clamp(min=eps) / w.clamp(min=eps))
        lh = torch.log(nh.clamp(min=eps) / h.clamp(min=eps))
        iw = (torch.minimum(gx2, nx2) - torch.maximum(gx1, nx1)).clamp(min=0.0)
        ih = (torch.minimum(gy2, ny2) - torch.maximum(gy1, ny1)).clamp(min=0.0)
        inter = iw * ih
        iou = inter / (w * h + nw * nh - inter + 1e-6)
        pair_geo = torch.stack([dx, dy, lw, lh, iou, nn_dist.clamp(max=2.0)], dim=-1)
        rel = self.rel_mlp(ent + ent_j + self.pair_proj(pair_geo.to(tokens.dtype)))

        # ---- assemble fixed-position slots ----------------------------------
        cand = torch.cat([ent, attr, act, rel], dim=1)                    # [B, 28, d]
        mask = torch.cat([valid, valid, valid[:, :A], rel_valid], dim=1)  # [B, 28]
        cand = cand * mask.unsqueeze(-1).to(cand.dtype)
        return cand, mask


# ---------------------------------------------------------------------------
# Layer 2 - Typed Slot Encoding
# ---------------------------------------------------------------------------

class TypedSlotEncoding(nn.Module):
    def __init__(self, dim: int, num_slots: int) -> None:
        super().__init__()
        self.slot_emb = nn.Parameter(torch.empty(num_slots, dim))
        self.type_emb = nn.Embedding(4, dim)   # entity / attribute / action / relation
        self.proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
        nn.init.normal_(self.slot_emb, std=0.02)
        nn.init.normal_(self.type_emb.weight, std=0.02)

    def forward(self, cand: Tensor, mask: Tensor, type_ids: Tensor) -> Tensor:
        x = cand + self.slot_emb + self.type_emb(type_ids)
        x = self.norm(self.proj(x))
        return x * mask.unsqueeze(-1).to(x.dtype)


# ---------------------------------------------------------------------------
# Layer 3 - Role Binding
# ---------------------------------------------------------------------------

class RoleBinding(nn.Module):
    def __init__(self, dim: int, attn_dim: int, num_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.attn = MaskedMHA(dim, attn_dim, num_heads, dropout)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x: Tensor, mask: Tensor) -> Tensor:
        x = self.norm(x + self.attn(x, x, mask))
        return x * mask.unsqueeze(-1).to(x.dtype)


# ---------------------------------------------------------------------------
# Layer 4 - Reliability Weighting
# ---------------------------------------------------------------------------

class ReliabilityWeighting(nn.Module):
    def __init__(self, dim: int, hidden: int) -> None:
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, 1))

    def forward(self, x: Tensor, mask: Tensor):
        r = torch.sigmoid(self.net(x)) * mask.unsqueeze(-1).to(x.dtype)   # [B, 28, 1]
        return x * r, r


# ---------------------------------------------------------------------------
# Layer 5 - Visual-Semantic Cross-Attention
# ---------------------------------------------------------------------------

class VisualSemanticCrossAttention(nn.Module):
    def __init__(self, dim: int, attn_dim: int, num_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.q_norm = nn.LayerNorm(dim)
        self.attn = MaskedMHA(dim, attn_dim, num_heads, dropout)

    def forward(self, tokens: Tensor, slots: Tensor, mask: Tensor, return_weights: bool = False):
        return self.attn(self.q_norm(tokens), slots, mask, return_weights)


# ---------------------------------------------------------------------------
# Layer 6 - Gated Residual Fusion
# ---------------------------------------------------------------------------

class GatedResidualFusion(nn.Module):
    """out = tokens + tanh(g) * ctx. With gate_init=0 the bridge starts as an exact
    identity, so training begins from the E0-R baseline representation."""

    def __init__(self, dim: int, gate_init: float = 0.0) -> None:
        super().__init__()
        self.gate = nn.Parameter(torch.full((dim,), float(gate_init)))

    def forward(self, tokens: Tensor, ctx: Tensor) -> Tensor:
        return tokens + torch.tanh(self.gate) * ctx


# ---------------------------------------------------------------------------
# Full bridge
# ---------------------------------------------------------------------------

class EAARNLUBridge(nn.Module):
    """[B, 197, dim] visual tokens + detections -> [B, 197, dim] enriched tokens."""

    def __init__(
        self,
        dim: int = 768,
        num_heads: int = 4,
        num_labels: int = 91,            # COCO category ids 0..90 (RF-DETR class_id)
        max_entities: int = 8,
        num_action_slots: int = 4,
        grid_size: int = 14,
        attn_dim: int = 480,             # inner width of both attention layers
        bottleneck_dim: int = 128,       # attribute / action / relation candidate MLPs
        reliability_hidden: int = 192,
        gate_init: float = 0.0,
        dropout: float = 0.0,
        use_role_binding: bool = True,   # layer 3; False -> skipped
        use_reliability: bool = True,    # layer 4; False -> every valid slot gets weight 1
    ) -> None:
        super().__init__()
        self.use_role_binding = use_role_binding
        self.use_reliability = use_reliability
        self.layer1 = SemanticEvidenceConstruction(
            dim, num_labels, max_entities, num_action_slots, grid_size, bottleneck_dim
        )
        n_slots = self.layer1.num_slots
        self.layer2 = TypedSlotEncoding(dim, n_slots)
        self.layer3 = RoleBinding(dim, attn_dim, num_heads, dropout) if use_role_binding else None
        self.layer4 = ReliabilityWeighting(dim, reliability_hidden) if use_reliability else None
        self.layer5 = VisualSemanticCrossAttention(dim, attn_dim, num_heads, dropout)
        self.layer6 = GatedResidualFusion(dim, gate_init)

        type_ids = torch.cat([
            torch.full((max_entities,), SLOT_ENTITY),
            torch.full((max_entities,), SLOT_ATTRIBUTE),
            torch.full((num_action_slots,), SLOT_ACTION),
            torch.full((max_entities,), SLOT_RELATION),
        ])
        self.register_buffer("type_ids", type_ids, persistent=False)
        self.num_slots = n_slots

    def forward(self, tokens: Tensor, det: Detections, return_aux: bool = False):
        cand, mask = self.layer1(tokens, det)                    # [B, 28, d], [B, 28]
        slots = self.layer2(cand, mask, self.type_ids)           # [B, 28, d]
        if self.layer3 is not None:
            slots = self.layer3(slots, mask)                     # [B, 28, d]
        if self.layer4 is not None:
            slots, reliability = self.layer4(slots, mask)        # [B, 28, d], [B, 28, 1]
        else:
            reliability = mask.unsqueeze(-1).to(slots.dtype)     # fixed weight 1 on valid slots
        if return_aux:
            ctx, xattn = self.layer5(tokens, slots, mask, return_weights=True)
        else:
            ctx = self.layer5(tokens, slots, mask)               # [B, 197, d]
        out = self.layer6(tokens, ctx)                           # [B, 197, d]
        if not return_aux:
            return out
        return out, {
            "slot_mask": mask,              # [B, 28]
            "slot_types": self.type_ids,    # [28]
            "reliability": reliability,     # [B, 28, 1]
            "cross_attn": xattn,            # [B, heads, 197, 28]
        }

    def layer_param_counts(self) -> dict:
        return {
            name: (sum(p.numel() for p in getattr(self, name).parameters())
                   if getattr(self, name) is not None else 0)
            for name in ("layer1", "layer2", "layer3", "layer4", "layer5", "layer6")
        }

Overwriting /kaggle/working/image_captioning/src/models/eaar_nlu.py


In [51]:
%%writefile /kaggle/working/image_captioning/src/models/e1_eaar.py
"""
E1-EAAR  -  E0-R captioner + EAAR-NLU V1 bridge
================================================
Subclasses E0RCaptioner and overrides only ``encode_image`` (plus the thin
``forward`` / ``generate`` wrappers so ``detections`` can be passed through).
The BART decoder, adapters and CLIP path are inherited unchanged.

The bridge is applied to the E0-R encoder output (after ``patch_proj`` /
``cls_proj``), i.e. exactly the [B, 197, 768] memory the decoder already reads.
With ``gate_init=0`` the bridge is an identity at step 0, so a loaded E0-R
checkpoint reproduces its baseline captions before any bridge training.
"""

from __future__ import annotations

from typing import Optional

import torch
from transformers.modeling_outputs import BaseModelOutput

from src.models.e0r import E0RCaptioner
from src.models.eaar_nlu import Detections, EAARNLUBridge

def _set_requires_grad(module, flag: bool) -> None:
    for p in module.parameters():
        p.requires_grad_(flag)


try:  # only needed if you train through the repo's Trainer
    from src.training.freeze import FreezeScheduler
except Exception:  # pragma: no cover
    FreezeScheduler = None


class E1EAARCaptioner(E0RCaptioner):
    def __init__(self, *args, bridge_kwargs: Optional[dict] = None, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.bridge_kwargs = dict(bridge_kwargs or {})
        self.bridge = EAARNLUBridge(dim=self.proj_dim, **self.bridge_kwargs)

    # ------------------------------------------------------------------
    def encode_image(
        self,
        pixel_values: torch.Tensor,
        detections: Optional[Detections] = None,
    ) -> torch.Tensor:
        tokens = super().encode_image(pixel_values)          # [B, 197, proj_dim]
        if detections is None:                               # baseline fallback
            return tokens
        return self.bridge(tokens, detections)               # [B, 197, proj_dim]

    # ------------------------------------------------------------------
    def forward(
        self,
        pixel_values: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        decoder_input_ids: Optional[torch.Tensor] = None,
        decoder_attention_mask: Optional[torch.Tensor] = None,
        detections: Optional[Detections] = None,
    ) -> dict:
        encoder_hidden = self.encode_image(pixel_values, detections)
        out = self.bart(
            encoder_outputs=BaseModelOutput(last_hidden_state=encoder_hidden),
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            labels=labels,
        )
        return {"loss": out.loss, "logits": out.logits}

    @torch.no_grad()
    def generate(
        self,
        pixel_values: torch.Tensor,
        detections: Optional[Detections] = None,
        **generate_kwargs,
    ) -> torch.Tensor:
        encoder_hidden = self.encode_image(pixel_values, detections)
        return self.bart.generate(
            encoder_outputs=BaseModelOutput(last_hidden_state=encoder_hidden),
            **generate_kwargs,
        )

    # ------------------------------------------------------------------
    def freeze_all_but_bridge(self) -> None:
        _set_requires_grad(self, False)
        _set_requires_grad(self.bridge, True)

    def get_model_config(self) -> dict:
        cfg = super().get_model_config()
        cfg["bridge_kwargs"] = self.bridge_kwargs
        return cfg


if FreezeScheduler is not None:

    class BridgeFreezeScheduler(FreezeScheduler):
        """Drop-in replacement for FreezeScheduler: every stage trains only the bridge.

        The stock scheduler re-enables adapters, projection heads and BART parts at
        each stage, which would silently un-freeze the E0-R weights.
        """

        def _apply_freeze(self, stage: dict) -> None:
            self.model.freeze_all_but_bridge()

Overwriting /kaggle/working/image_captioning/src/models/e1_eaar.py


In [52]:
%%writefile /kaggle/working/image_captioning/tests/test_eaar_nlu.py
import torch

from src.models.eaar_nlu import Detections, EAARNLUBridge, detections_to_tensors

B, K = 3, 8


def make_batch(n_valid=(8, 3, 0), seed=0):
    g = torch.Generator().manual_seed(seed)
    tokens = torch.randn(B, 197, 768, generator=g)
    xy = torch.rand(B, K, 2, generator=g) * 0.6
    wh = 0.1 + torch.rand(B, K, 2, generator=g) * 0.3
    boxes = torch.cat([xy, xy + wh], dim=-1)
    labels = torch.randint(1, 91, (B, K), generator=g)
    scores = torch.rand(B, K, generator=g)
    valid = torch.zeros(B, K, dtype=torch.bool)
    for i, n in enumerate(n_valid):
        valid[i, :n] = True
    return tokens, Detections(boxes, labels, scores, valid)


def test_shapes_and_aux():
    bridge = EAARNLUBridge()
    tokens, det = make_batch()
    out, aux = bridge(tokens, det, return_aux=True)
    assert out.shape == (B, 197, 768)
    assert aux["reliability"].shape == (B, 28, 1)
    assert aux["cross_attn"].shape == (B, 4, 197, 28)
    assert aux["slot_mask"].shape == (B, 28)
    assert torch.isfinite(out).all()


def test_identity_at_init():
    bridge = EAARNLUBridge(gate_init=0.0)
    tokens, det = make_batch()
    assert torch.allclose(bridge(tokens, det), tokens)


def test_empty_image_passes_through_even_with_open_gate():
    bridge = EAARNLUBridge(gate_init=0.5)
    tokens, det = make_batch(n_valid=(0, 0, 0))
    out = bridge(tokens, det)
    assert torch.isfinite(out).all()
    assert torch.allclose(out, tokens)


def test_padded_entries_do_not_leak():
    bridge = EAARNLUBridge(gate_init=0.5).eval()
    tokens, det = make_batch()
    ref = bridge(tokens, det)
    junk = Detections(
        torch.where(det.valid[..., None], det.boxes, torch.rand_like(det.boxes)),
        torch.where(det.valid, det.labels, torch.randint(1, 91, det.labels.shape)),
        torch.where(det.valid, det.scores, torch.rand_like(det.scores)),
        det.valid,
    )
    assert torch.allclose(bridge(tokens, junk), ref, atol=1e-5)


def test_gradients_reach_every_layer():
    bridge = EAARNLUBridge(gate_init=0.1)
    tokens, det = make_batch()
    bridge(tokens, det).sum().backward()
    for name, p in bridge.named_parameters():
        assert p.grad is not None and torch.isfinite(p.grad).all(), name


def test_param_count_report():
    bridge = EAARNLUBridge()
    total = sum(p.numel() for p in bridge.parameters() if p.requires_grad)
    print(bridge.layer_param_counts(), "total:", f"{total:,}")
    assert 4_000_000 < total < 4_800_000


def test_detection_adapter_center_crop():
    # 640x480 image -> short side 480 scaled to 224, 37.33px cropped from each side
    d = detections_to_tensors(
        xyxy=[[0, 0, 100, 100], [200, 100, 440, 380]],
        confidence=[0.9, 0.8],
        class_id=[1, 18],
        image_size=(640, 480),
    )
    assert d.valid.tolist() == [True, True] + [False] * 6
    assert d.boxes.min() >= 0 and d.boxes.max() <= 1
    # first box lies mostly in the cropped-away left strip: x1 clipped to 0
    assert d.boxes[0, 0] == 0


LITE = dict(use_role_binding=False, use_reliability=False)


def test_three_part_variant():
    bridge = EAARNLUBridge(**LITE)
    tokens, det = make_batch()
    assert torch.allclose(bridge(tokens, det), tokens)            # identity at init
    out, aux = EAARNLUBridge(gate_init=0.3, **LITE)(tokens, det, return_aux=True)
    assert out.shape == (B, 197, 768) and torch.isfinite(out).all()
    assert aux["reliability"].shape == (B, 28, 1)
    total = sum(p.numel() for p in bridge.parameters())
    assert total == 2_770_464, total
    counts = bridge.layer_param_counts()
    assert counts["layer3"] == 0 and counts["layer4"] == 0


def test_three_part_gradients():
    bridge = EAARNLUBridge(gate_init=0.1, **LITE)
    tokens, det = make_batch()
    bridge(tokens, det).sum().backward()
    for name, p in bridge.named_parameters():
        assert p.grad is not None and torch.isfinite(p.grad).all(), name

Overwriting /kaggle/working/image_captioning/tests/test_eaar_nlu.py


In [53]:
!cd /kaggle/working/image_captioning && python -m pytest tests/test_eaar_nlu.py -q

.........                                                                [100%]
9 passed in 8.42s


## Data + detection cache
Reuses your existing cache if it is in `/kaggle/input` or in `outputs/e1_eaar/`; otherwise runs RF-DETR (about 16 min). **Save the cache file as a Kaggle dataset afterwards.**

In [ ]:
import csv, random, math, time
from collections import defaultdict
from pathlib import Path
import numpy as np, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm.auto import tqdm
from transformers import CLIPProcessor, BartTokenizer
from src.models.eaar_nlu import Detections, EAARNLUBridge, detections_to_tensors
from src.models.e1_eaar import E1EAARCaptioner

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = AMP and device == "cuda"

import yaml
cfg_path = Path("configs/e0r.yaml")
m_cfg = yaml.safe_load(cfg_path.read_text())["model"] if cfg_path.exists() else {}
CLIP_NAME = m_cfg.get("clip_model_name", "openai/clip-vit-base-patch16")
BART_NAME = m_cfg.get("bart_model_name", "facebook/bart-base")
processor = CLIPProcessor.from_pretrained(CLIP_NAME)
tokenizer = BartTokenizer.from_pretrained(BART_NAME)


class CaptionDetDataset(Dataset):
    def __init__(self, split, limit=None, max_length=64, max_captions=5, seed=42):
        self.image_dir, self.split, self.max_length = Path(IMAGE_DIR), split, max_length
        caps = defaultdict(list)
        with open(ANN_FILE, "r", encoding="utf-8-sig", newline="") as f:
            for row in csv.reader(f):
                if len(row) < 2: continue
                fn, cap = row[0].strip(), ",".join(row[1:]).strip()
                if fn.lower() in ("image", "image_name", "filename"): continue
                if fn and cap: caps[fn].append(cap)
        # same split as scripts/evaluate.py: sorted unique names -> shuffle(seed 42) -> 80/20
        samples = [(fn, caps[fn][:max_captions]) for fn in sorted(caps) if (self.image_dir / fn).exists()]
        random.Random(seed).shuffle(samples)
        k = int(0.8 * len(samples))
        self.samples = samples[:k] if split == "train" else samples[k:]
        if limit: self.samples = self.samples[:limit]
        self.det_cache = None
        print(f"{split}: {len(self.samples)} images")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fn, caps = self.samples[idx]
        img = Image.open(self.image_dir / fn).convert("RGB")
        pv = processor(images=img, return_tensors="pt").pixel_values.squeeze(0)
        cap = random.choice(caps) if self.split == "train" else caps[0]
        enc = tokenizer(cap, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        ids, am = enc.input_ids.squeeze(0), enc.attention_mask.squeeze(0)
        labels = ids.clone(); labels[am == 0] = -100
        if self.det_cache is not None and fn in self.det_cache:
            b, l, s, v = self.det_cache[fn]
        else:
            e = Detections.empty(8); b, l, s, v = e.boxes, e.labels, e.scores, e.valid
        return {"pixel_values": pv, "labels": labels, "decoder_attention_mask": am, "image_id": fn,
                "det_boxes": b, "det_labels": l, "det_scores": s, "det_valid": v}


train_ds = CaptionDetDataset("train", limit=N_TRAIN)
val_ds   = CaptionDetDataset("val",   limit=N_VAL)
assert not ({f for f, _ in train_ds.samples} & {f for f, _ in val_ds.samples}), "train/val overlap!"
if not SMOKE_TEST: assert len(val_ds) == 1619, f"expected the peers' 1619 val images, got {len(val_ds)}"
needed = sorted({fn for ds in (train_ds, val_ds) for fn, _ in ds.samples})

# ---- find or build the detection cache ----
cands = glob.glob("/kaggle/input/**/*dets*.pt", recursive=True) + [f"{REPO}/outputs/e1_eaar/flickr8k_dets.pt", "/kaggle/working/det_cache.pt"]
DET_CACHE = next((p for p in cands if os.path.exists(p)), "/kaggle/working/det_cache.pt")
cache = torch.load(DET_CACHE) if os.path.exists(DET_CACHE) else {}
todo = [fn for fn in needed if fn not in cache]
print(f"cache file: {DET_CACHE} | {len(cache)} cached | {len(todo)} still to detect")
if todo:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "rfdetr", "supervision"], check=False)
    from rfdetr import RFDETRSmall
    detector = RFDETRSmall()
    DET_CACHE = "/kaggle/working/det_cache.pt"
    for n, fn in enumerate(tqdm(todo)):
        img = Image.open(f"{IMAGE_DIR}/{fn}").convert("RGB")
        d = detector.predict(img, threshold=0.3)
        det = detections_to_tensors(d.xyxy if d.xyxy is not None else [], d.confidence if d.confidence is not None else [],
                                    d.class_id if d.class_id is not None else [], img.size, max_det=8)
        cache[fn] = (det.boxes, det.labels, det.scores, det.valid)
        if (n + 1) % 500 == 0: torch.save(cache, DET_CACHE)
    torch.save(cache, DET_CACHE); del detector; torch.cuda.empty_cache()
for ds in (train_ds, val_ds): ds.det_cache = cache
print("mean detections/image:", np.mean([int(cache[fn][3].sum()) for fn in needed]))

train: 6472 images
val: 1619 images
cache file: /kaggle/working/det_cache.pt | 1000 cached | 7091 still to detect
[2026-11-24 20:00:17] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-small.pth already exists with correct MD5 hash.
[2026-11-24 20:00:19] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-small.pth already exists with correct MD5 hash.


  0%|          | 0/7091 [00:00<?, ?it/s]

[2026-11-24 20:00:41] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. Create predictor = native_model.inference(...) and use predictor.predict(...). For a checkpoint, first load native_model = RFDETR.from_checkpoint(...).


## Build the model, load E0-R (strict check), freeze everything but the bridge

In [ ]:
model = E1EAARCaptioner(clip_model_name=CLIP_NAME, bart_model_name=BART_NAME,
                        proj_dim=m_cfg.get("proj_dim", 768),
                        adapter_bottleneck_dim=m_cfg.get("adapter_bottleneck_dim", 64),
                        bridge_kwargs=LITE)
ckpt = torch.load(E0R_CKPT, map_location="cpu", weights_only=False)
assert "model_state" in ckpt, f"unexpected checkpoint keys: {list(ckpt)[:8]}"
missing, unexpected = model.load_state_dict(ckpt["model_state"], strict=False)
bad = [k for k in missing if not k.startswith("bridge.")]
print("E0-R checkpoint epoch:", ckpt.get("epoch"), "| missing non-bridge:", bad, "| unexpected:", list(unexpected))
assert not bad and not unexpected, "E0-R weights did not load cleanly."
model.to(device)
model.eval()

def reset_bridge(kwargs, seed=SEED):
    """Fresh bridge with identical init for every variant."""
    torch.manual_seed(seed)
    model.bridge = EAARNLUBridge(dim=768, **kwargs).to(device)
    model.bridge_kwargs = dict(kwargs)
    model.freeze_all_but_bridge()
    return sum(p.numel() for p in model.bridge.parameters() if p.requires_grad)

print("3-part bridge trainable params:", f"{reset_bridge(LITE):,}", "(expected 2,770,464)")

In [ ]:
import os, zipfile, subprocess
SRC = "/kaggle/input/datasets/fihsak/ckpt-epoch05-pt/ckpt_epoch05"
out = "/kaggle/working/ckpt_epoch05.pt"
if os.path.exists(out): os.remove(out)
r = subprocess.run(f"cd {os.path.dirname(SRC)} && zip -0 -r -X -q {out} ckpt_epoch05", shell=True, capture_output=True, text=True)
print("zip rc:", r.returncode, r.stderr[:300])
print("entries:", len(zipfile.ZipFile(out).namelist()), f"| {os.path.getsize(out)/1e6:.0f} MB")
import torch
ck = torch.load(out, map_location="cpu", weights_only=False)
print("keys:", list(ck)[:8], "| epoch:", ck.get("epoch"))

## Helpers: generation, metrics (with per-image CIDEr), paired bootstrap

In [ ]:
def make_det(batch, mode="real"):
    t = [batch["det_boxes"], batch["det_labels"], batch["det_scores"], batch["det_valid"]]
    if mode == "shuffled":                      # give every image a different image's detections
        t = [torch.roll(x, 1, 0) for x in t]
    return Detections(*t).to(device)

@torch.no_grad()
def generate_captions(ds, det_mode="real", bs=16):
    """det_mode: 'none' (baseline, bridge bypassed) | 'real' | 'shuffled'."""
    loader = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=2)
    model.eval(); preds = {}
    for batch in tqdm(loader, leave=False):
        pv = batch["pixel_values"].to(device)
        det = None if det_mode == "none" else make_det(batch, det_mode)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            ids = model.generate(pv, detections=det, **GEN_KW)
        for fid, t in zip(batch["image_id"], tokenizer.batch_decode(ids, skip_special_tokens=True)):
            preds[fid] = t.strip()
    return preds

REFS = dict(val_ds.samples)
# no text normalisation: same raw strings as src/eval/metrics.py, so numbers are comparable with the E0-R results

def score(preds):
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.cider.cider import Cider
    from pycocoevalcap.rouge.rouge import Rouge
    ids = list(preds)
    gts = {k: list(REFS[k]) for k in ids}
    res = {k: [preds[k]] for k in ids}
    b, _ = Bleu(4).compute_score(gts, res, verbose=0)
    c, c_img = Cider().compute_score(gts, res)
    r, _ = Rouge().compute_score(gts, res)
    return {"BLEU-4": b[3], "CIDEr": c, "ROUGE-L": r}, dict(zip(ids, c_img))

def paired_bootstrap(a, b, n=2000, seed=0):
    """95% CI of mean(a - b) over images (a, b: dict id -> per-image score)."""
    ids = sorted(set(a) & set(b)); d = np.array([a[i] - b[i] for i in ids])
    rng = np.random.default_rng(seed)
    means = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(n)])
    return d.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

## Identity check (must pass): untrained bridge == baseline

In [ ]:
tiny = Subset(val_ds, range(16))
a = generate_captions(tiny, "none"); b = generate_captions(tiny, "real")
same = sum(a[k] == b[k] for k in a); print(f"identical captions: {same}/{len(a)}")
assert same == len(a), "Bridge is not an identity at init. Do not train."

## Baseline (E0-R, no detections): full val set and progress-curve subset

In [ ]:
t0 = time.time()
base_preds = generate_captions(val_ds, "none")
base_scores, base_cider_img = score(base_preds)
curve_ids = [fn for fn, _ in val_ds.samples[:N_CURVE]]
base_curve = score({k: base_preds[k] for k in curve_ids})[0]
curve_ds = Subset(val_ds, range(N_CURVE))
print("BASELINE:", {k: round(v, 4) for k, v in base_scores.items()}, f"[{time.time()-t0:.0f}s]")
# ---- does our baseline reproduce the peers' E0-R numbers? ----
if not SMOKE_TEST:
    for k, v in PEER_REF.items():
        print(f"  {k:8s} ours {base_scores[k]:.4f} | peers {v:.4f} | diff {base_scores[k]-v:+.4f}")
    assert abs(base_scores["CIDEr"] - PEER_REF["CIDEr"]) < 0.01, "Baseline does not match the peers' E0-R CIDEr: check split / checkpoint / decoding before trusting any delta."

## Train each variant (bridge only), evaluating after every epoch for the progress curve

In [ ]:
from transformers import get_cosine_schedule_with_warmup
RESULTS = {}

def train_variant(name, kwargs, train_mode):
    n_params = reset_bridge(kwargs)
    loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    named = list(model.bridge.named_parameters())
    gate = [p for n, p in named if n == "layer6.gate"]
    decay = [p for n, p in named if p.ndim >= 2]; no_decay = [p for n, p in named if p.ndim < 2 and n != "layer6.gate"]
    params = decay + no_decay + gate
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY, "lr": LR},
                             {"params": no_decay, "weight_decay": 0.0, "lr": LR},
                             {"params": gate, "weight_decay": 0.0, "lr": GATE_LR}])
    total = MAX_OPT_STEPS or (len(loader) // ACCUM) * EPOCHS
    sched = get_cosine_schedule_with_warmup(opt, min(WARMUP, max(1, total // 10)), total)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    print(f"\n=== {name}: {n_params:,} params, train detections: {train_mode}, {total} optimizer steps ===")
    hist = [{"epoch": 0, "step": 0, "loss": None, "gate": 0.0, **base_curve}]
    step, done = 0, False
    for epoch in range(1, EPOCHS + 1):
        model.eval(); model.bridge.train()
        opt.zero_grad(set_to_none=True); run, n = 0.0, 0
        for i, batch in enumerate(tqdm(loader, desc=f"{name} epoch {epoch}")):
            pv = batch["pixel_values"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            am = batch["decoder_attention_mask"].to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                loss = model(pixel_values=pv, labels=labels, decoder_attention_mask=am,
                             detections=make_det(batch, train_mode))["loss"]
            scaler.scale(loss / ACCUM).backward(); run += loss.item(); n += 1
            if (i + 1) % ACCUM == 0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, MAX_GRAD)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
                if step % 50 == 0 or step == 1:
                    print(f"  step {step:5d} loss {run/n:.4f} gate|mean| {torch.tanh(model.bridge.layer6.gate).abs().mean().item():.4f}")
                if MAX_OPT_STEPS and step >= MAX_OPT_STEPS: done = True; break
        sc_real = score(generate_captions(curve_ds, "real"))[0]
        hist.append({"epoch": epoch, "step": step, "loss": run / max(n, 1),
                     "gate": torch.tanh(model.bridge.layer6.gate).abs().mean().item(), **sc_real})
        print(f"  epoch {epoch}: curve-subset {({k: round(v,4) for k,v in sc_real.items()})}")
        torch.save({"bridge": model.bridge.state_dict(), "kwargs": kwargs, "epoch": epoch, "e0r_ckpt": E0R_CKPT},
                   f"{OUT_DIR}/{name}_epoch{epoch:02d}.pt")
        if done: break

    preds = generate_captions(val_ds, "real");      s_real, c_real = score(preds)
    p_shuf = generate_captions(val_ds, "shuffled"); s_shuf, _ = score(p_shuf)
    RESULTS[name] = dict(params=n_params, hist=hist, final=s_real, cider_img=c_real, preds=preds, shuffled_eval=s_shuf)
    print(f"  FINAL {name}:", {k: round(v, 4) for k, v in s_real.items()},
          "| test-time shuffled:", {k: round(v, 4) for k, v in s_shuf.items()})

for name, (kw, mode) in VARIANTS.items():
    train_variant(name, kw, mode)

## Results table (this is what you show)

In [ ]:
import json
import pandas as pd
rows = [{"Model": "E0-R baseline (no detections)", "Params trained": 0, **base_scores, "dCIDEr vs baseline [95% CI]": "-"}]
labels = {"3part": "3-part bridge (L1+L2+L5), real detections",
          "3part_shuffled_train": "control: 3-part trained on SHUFFLED detections",
          "full6": "full 6-layer bridge"}
for name, r in RESULTS.items():
    d, lo, hi = paired_bootstrap(r["cider_img"], base_cider_img)
    rows.append({"Model": labels.get(name, name), "Params trained": r["params"], **r["final"],
                 "dCIDEr vs baseline [95% CI]": f"{d:+.4f} [{lo:+.4f}, {hi:+.4f}]"})
    if name != "3part_shuffled_train":
        rows.append({"Model": f"   same weights, detections shuffled at test time", "Params trained": r["params"],
                     **r["shuffled_eval"], "dCIDEr vs baseline [95% CI]": "-"})
df = pd.DataFrame(rows)
for c in ("BLEU-4", "CIDEr", "ROUGE-L"): df[c] = df[c].round(4)
display(df)
df.to_csv(f"{OUT_DIR}/results.csv", index=False)
json.dump({"baseline": base_scores, "variants": {k: {"params": v["params"], "final": v["final"], "shuffled_eval": v["shuffled_eval"],
           "hist": v["hist"]} for k, v in RESULTS.items()}, "smoke_test": SMOKE_TEST, "n_val": len(val_ds)},
          open(f"{OUT_DIR}/results.json", "w"), indent=1, default=float)

if "3part" in RESULTS:
    d, lo, hi = paired_bootstrap(RESULTS["3part"]["cider_img"], base_cider_img)
    verdict = "CI excludes 0 -> improvement is statistically supported" if lo > 0 else "CI includes 0 -> not distinguishable from noise yet"
    print(f"\n3-part vs baseline: dCIDEr {d:+.4f} (95% CI {lo:+.4f} to {hi:+.4f}) -> {verdict}")
    drop = RESULTS["3part"]["final"]["CIDEr"] - RESULTS["3part"]["shuffled_eval"]["CIDEr"]
    print(f"Shuffling detections at test time changes CIDEr by {-drop:+.4f} (a clear drop means the model relies on grounded evidence)")
if SMOKE_TEST: print("\nSMOKE_TEST: tiny run, do NOT quote these numbers.")

## Figures: progress curve, bar chart, qualitative examples

In [ ]:
import matplotlib.pyplot as plt, textwrap
# ---- progress curve (CIDEr on the first N_CURVE val images, per epoch; epoch 0 = baseline) ----
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for name, r in RESULTS.items():
    ax[0].plot([h["epoch"] for h in r["hist"]], [h["CIDEr"] for h in r["hist"]], marker="o", label=labels.get(name, name))
    ax[1].plot([h["epoch"] for h in r["hist"]], [h["BLEU-4"] for h in r["hist"]], marker="o", label=labels.get(name, name))
for a, t in zip(ax, ("CIDEr", "BLEU-4")):
    a.axhline(base_curve[t], ls="--", c="gray", label="E0-R baseline"); a.set_xlabel("epoch of bridge training"); a.set_title(f"{t} vs training"); a.grid(alpha=.3)
ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/progress_curve.png", dpi=160); plt.show()

# ---- bar chart ----
names = ["E0-R baseline"] + [labels.get(n, n).split(",")[0] for n in RESULTS]
vals = [base_scores["CIDEr"]] + [r["final"]["CIDEr"] for r in RESULTS.values()]
fig, ax = plt.subplots(figsize=(7, 3.4)); bars = ax.bar(range(len(vals)), vals, color=["gray"] + ["tab:blue"] * len(RESULTS))
ax.set_xticks(range(len(vals))); ax.set_xticklabels([textwrap.fill(n, 18) for n in names], fontsize=7)
ax.set_ylabel("CIDEr"); ax.set_ylim(min(vals) * 0.9, max(vals) * 1.05)
for b, v in zip(bars, vals): ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/results_bar.png", dpi=160); plt.show()

# ---- qualitative: biggest per-image gains (and 2 regressions, for honesty) ----
if "3part" in RESULTS:
    r = RESULTS["3part"]; diff = {k: r["cider_img"][k] - base_cider_img[k] for k in r["cider_img"]}
    order = sorted(diff, key=diff.get); picks = order[::-1][:4] + order[:2]
    fig, axs = plt.subplots(2, 3, figsize=(14, 8))
    for ax_, fn in zip(axs.ravel(), picks):
        ax_.imshow(Image.open(f"{IMAGE_DIR}/{fn}").convert("RGB")); ax_.axis("off")
        ax_.set_title(f"dCIDEr {diff[fn]:+.2f}", fontsize=9, color="green" if diff[fn] > 0 else "red")
        ax_.text(0, -0.04, "base: " + textwrap.fill(base_preds[fn], 45) + "\nbridge: " + textwrap.fill(r["preds"][fn], 45),
                 transform=ax_.transAxes, va="top", fontsize=8)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/qualitative.png", dpi=160, bbox_inches="tight"); plt.show()
print("saved to", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))

## After the smoke run
1. Checks: 9 tests pass, identity 16/16, loss decreases, `gate|mean|` > 0. (Baseline match with the peers' numbers is only asserted on the full run.)
2. Set `SMOKE_TEST = False` and re-run from the config cell of Part 2.
3. Save `/kaggle/working/e1_outputs/` and the detection cache as Kaggle datasets so a session reset doesn't lose them.
4. Later, un-comment `"full6"` in `VARIANTS` to add the full 6-layer row.